# 01 — Explore the NBS PMS Price Watch files

**Exploration only** — nothing here is imported by `pipeline/`. Findings and the decisions they led to
are in [`docs/decisions.md`](../docs/decisions.md) (D-numbers are referenced below).

Input: `landing/raw/pms/<YYYY-MM>/` with one xlsx + one pdf per release. Re-run top to bottom whenever a
new release lands; any check that changes is a layout change worth recording.

In [1]:
import datetime as dt
import re
from decimal import ROUND_HALF_UP, Decimal
from hashlib import sha256
from pathlib import Path

import openpyxl
import pandas as pd
import pdfplumber

pd.set_option("display.width", 200, "display.max_columns", 30, "display.max_rows", 100)
ROOT = Path.cwd() if (Path.cwd() / "pyproject.toml").exists() else Path.cwd().parent
RAW = ROOT / "landing" / "raw" / "pms"
RELEASES = sorted(d for d in RAW.iterdir() if d.is_dir())
print(len(RELEASES), "releases:", RELEASES[0].name, "->", RELEASES[-1].name)

19 releases: 2024-11 -> 2026-05


## 1. Inventory

In [2]:
inv = pd.DataFrame([{"release": d.name, "file": f.name, "kb": f.stat().st_size // 1024,
                     "sha256_8": sha256(f.read_bytes()).hexdigest()[:8]}
                    for d in RELEASES for f in sorted(d.iterdir())])
print("duplicate hashes:", inv.sha256_8.duplicated().sum())
inv

duplicate hashes: 0


,release,file,kb,sha256_8
0,2024-11,PMS NOV 2024 REPORT.xlsx,14,c6ac797a
1,2024-11,PMS_Report_NOV_2024.pdf,626,51a65092
2,2024-12,PMS DECEMBER 2024.xlsx,16,86760185
3,2024-12,PMS_Report_DEC_2024.pdf,630,81da8bf2
4,2025-01,PMS_Jan_2025.xlsx,12,166a6872
5,2025-01,PMS_Report_JAN_2025.pdf,1061,0502ad71
6,2025-02,PMS_FEB_2025.xlsx,11,850a76dc
7,2025-02,PMS_Report_FEB_2025.pdf,1061,2d54399b
8,2025-03,PMS Report March 2025.xlsx,11,c7be2f3b
9,2025-03,PMS_Report_March_2025.pdf,466,c9406e40


## 2. Structure survey (D1, D2, D8, D11)
Header = the row whose B:D are three Excel dates. Everything else is located relative to it.

In [3]:
def is_date(v):
    return isinstance(v, dt.datetime)

def survey(xlsx):
    wf = openpyxl.load_workbook(xlsx)                      # formulas
    ws = openpyxl.load_workbook(xlsx, data_only=True).worksheets[0]  # cached values
    hdr = next(r for r in range(1, 41) if all(is_date(ws.cell(r, c).value) for c in (2, 3, 4)))
    col_a = {r: ws.cell(r, 1).value for r in range(1, ws.max_row + 1)}
    avg_rows = [r for r, v in col_a.items() if isinstance(v, str) and v.strip().upper() == "AVERAGE"]
    title = next((c.value.strip() for row in ws.iter_rows(max_row=hdr) for c in row
                  if isinstance(c.value, str) and "REPORT" in c.value.upper() and len(c.value) < 120), None)
    zone_at = next((c.coordinate for row in ws.iter_rows() for c in row
                    if isinstance(c.value, str) and c.value.strip() == "North Central"), None)
    formulas = sum(isinstance(c.value, str) and c.value.startswith("=") for row in wf.worksheets[0].iter_rows() for c in row)
    return {"file": xlsx.name, "sheet": ws.title, "header_row": hdr, "header_A": col_a[hdr],
            "months": "/".join(ws.cell(hdr, c).value.strftime("%b-%y") for c in (2, 3, 4)),
            "average_rows": avg_rows, "title_tail": title[-26:] if title else None,
            "zone_block_at": zone_at, "formulas": formulas, "external_links": len(wf._external_links)}

structure = pd.DataFrame([{"release": d.name, **survey(next(d.glob("*.xlsx")))} for d in RELEASES])
structure

,release,file,sheet,header_row,header_A,months,average_rows,title_tail,zone_block_at,formulas,external_links
0,2024-11,PMS NOV 2024 REPORT.xlsx,November 2024,2,State,Nov-23/Oct-24/Nov-24,[40],NaN,F4,0,0
1,2024-12,PMS DECEMBER 2024.xlsx,Fuel December 2024,1,State,Dec-23/Nov-24/Dec-24,[39],NaN,F3,13,1
2,2025-01,PMS_Jan_2025.xlsx,Fuel January 2025,1,State,Jan-24/Dec-24/Jan-25,[39],NaN,F3,0,0
3,2025-02,PMS_FEB_2025.xlsx,Fuel February 2025,1,State,Feb-24/Jan-25/Feb-25,[39],NaN,F3,0,0
4,2025-03,PMS Report March 2025.xlsx,Fuel March 2025,1,State,Mar-24/Feb-25/Mar-25,[39],NaN,F3,0,0
5,2025-04,Fuel_Report_April_2025.xlsx,Fuel April 2025,1,State,Apr-24/Mar-25/Apr-25,[39],NaN,F3,0,0
6,2025-05,Fuel_Report_May_2025.xlsx,Fuel May 2025,1,State,May-24/Apr-25/May-25,[39],NaN,F3,0,0
7,2025-06,PMS_Report_June_2025.xlsx,Fuel June 2025,1,State,Jun-24/May-25/Jun-25,[39],NaN,F3,0,0
8,2025-07,Fuel_Report_July_2025.xlsx,Fuel JULY 2025,1,State,Jul-24/Jun-25/Jul-25,[39],NaN,F3,165,1
9,2025-08,PMS AUGUST 2025.xlsx,PMS AUGUST 2025,2,State,Aug-24/Jul-25/Aug-25,[40],R LITRE AUGUST 2025 REPORT,F4,0,0


## 3. Cross-report consistency: does each report's M-1 column equal the previous report's M column? (D3, D7)
This is the check that exposed a locally edited Nov 2024 copy (column A shifted up one row: 0/36 states matched).
Label misalignment can't be caught by the national-mean check (a mean ignores labels), but it breaks this one.

In [4]:
def header_row(ws):
    return next(r for r in range(1, 41) if all(is_date(ws.cell(r, c).value) for c in (2, 3, 4)))

def read_state_block(d):
    ws = openpyxl.load_workbook(next(d.glob("*.xlsx")), data_only=True).worksheets[0]
    hdr = header_row(ws)
    if str(ws.cell(hdr, 1).value).strip().lower() != "state":
        raise ValueError(f"{d.name}: header row {hdr} col A is {ws.cell(hdr, 1).value!r}, expected 'State'")
    return {ws.cell(r, 1).value.strip(): (ws.cell(r, 3).value, ws.cell(r, 4).value) for r in range(hdr + 1, hdr + 38)}

blocks = {d.name: read_state_block(d) for d in RELEASES}
pairs = []
for prev, cur in zip(RELEASES, RELEASES[1:]):
    a, b = blocks[prev.name], blocks[cur.name]
    norm = lambda n: {"Nassarawa": "Nasarawa"}.get(n, n)
    a = {norm(k): v for k, v in a.items()}; b = {norm(k): v for k, v in b.items()}
    same = sum(abs(float(a[s][1]) - float(b[s][0])) < 0.005 for s in a)
    pairs.append({"previous": prev.name, "current": cur.name, "states_equal": f"{same}/37"})
pd.DataFrame(pairs)

,previous,current,states_equal
0,2024-11,2024-12,37/37
1,2024-12,2025-01,37/37
2,2025-01,2025-02,37/37
3,2025-02,2025-03,37/37
4,2025-03,2025-04,37/37
5,2025-04,2025-05,37/37
6,2025-05,2025-06,37/37
7,2025-06,2025-07,37/37
8,2025-07,2025-08,37/37
9,2025-08,2025-09,37/37


## 4. Load every file into one long table

In [5]:
ZONE = {
    "Abia": "South East", "Adamawa": "North East", "Akwa Ibom": "South South", "Anambra": "South East",
    "Bauchi": "North East", "Bayelsa": "South South", "Benue": "North Central", "Borno": "North East",
    "Cross River": "South South", "Delta": "South South", "Ebonyi": "South East", "Edo": "South South",
    "Ekiti": "South West", "Enugu": "South East", "Abuja": "North Central", "Gombe": "North East",
    "Imo": "South East", "Jigawa": "North West", "Kaduna": "North West", "Kano": "North West",
    "Katsina": "North West", "Kebbi": "North West", "Kogi": "North Central", "Kwara": "North Central",
    "Lagos": "South West", "Nasarawa": "North Central", "Niger": "North Central", "Ogun": "South West",
    "Ondo": "South West", "Osun": "South West", "Oyo": "South West", "Plateau": "North Central",
    "Rivers": "South South", "Sokoto": "North West", "Taraba": "North East", "Yobe": "North East",
    "Zamfara": "North West",
}
ALIAS = {"Nassarawa": "Nasarawa"}  # D10; FCT appears as "Abuja" throughout

prices, national, zones = [], [], []
for d in RELEASES:
    ws = openpyxl.load_workbook(next(d.glob("*.xlsx")), data_only=True).worksheets[0]
    hdr = header_row(ws)
    months = [ws.cell(hdr, c).value.date() for c in (2, 3, 4)]
    vintage = max(months)
    for r in range(hdr + 1, hdr + 38):
        raw = ws.cell(r, 1).value.strip()
        for c, m in zip((2, 3, 4), months):
            prices.append({"vintage": vintage, "month": m, "raw_name": raw, "state": ALIAS.get(raw, raw),
                           "price": float(ws.cell(r, c).value)})
    avg_rows = [r for r in range(hdr + 1, ws.max_row + 1) if str(ws.cell(r, 1).value).strip().upper() == "AVERAGE"]
    assert avg_rows == [hdr + 38], (d.name, avg_rows)  # AVERAGE directly after the 37 states, exactly once
    avg_r = avg_rows[0]
    national += [{"vintage": vintage, "month": m, "published": float(ws.cell(avg_r, c).value)} for c, m in zip((2, 3, 4), months)]
    for row in ws.iter_rows():
        for cell in row:
            v = ws.cell(cell.row, cell.column + 1).value
            if isinstance(cell.value, str) and cell.value.strip() in set(ZONE.values()) and isinstance(v, (int, float)):
                zones.append({"vintage": vintage, "zone": cell.value.strip(), "published": float(v)})

P, N, Z = pd.DataFrame(prices), pd.DataFrame(national), pd.DataFrame(zones)
print(len(P), "observations |", P.vintage.nunique(), "vintages |", P.month.nunique(), "months:", P.month.min(), "->", P.month.max())
print("37 states in every (vintage, month):", P.groupby(["vintage", "month"]).state.nunique().eq(37).all())
print("unknown names:", sorted(set(P.state) - set(ZONE)) or "none")
P.groupby("raw_name").vintage.agg(["count", "min", "max"]).query("raw_name in ['Abuja', 'Nassarawa', 'Nasarawa']")

2109 observations | 19 vintages | 31 months: 2023-11-01 -> 2026-05-01
37 states in every (vintage, month): True
unknown names: none


,count,min,max
raw_name,,,
Abuja,57,2024-11-01,2026-05-01
Nasarawa,30,2025-08-01,2026-05-01
Nassarawa,27,2024-11-01,2025-07-01


## 5. National and zone averages vs mean of states (D6)

In [6]:
nat = P.groupby(["vintage", "month"]).price.mean().rename("mean").reset_index().merge(N, on=["vintage", "month"])
print(f"national: {len(nat)} vintage-months, max |published - mean| = {(nat.published - nat['mean']).abs().max():.2e}")

cur = (P[P.month == P.vintage].assign(zone=lambda d: d.state.map(ZONE))
       .groupby(["vintage", "zone"]).price.mean().rename("mean").reset_index())
zc = Z.merge(cur, on=["vintage", "zone"]).assign(diff=lambda d: (d.published - d["mean"]).abs())
print(f"zones: {len(zc)} published figures, {int((zc['diff'] < 0.01).sum())} match")
zc[zc["diff"] >= 0.01]

national: 57 vintage-months, max |published - mean| = 4.55e-12
zones: 114 published figures, 113 match


,vintage,zone,published,mean,diff
13,2025-01-01,North Central,1307.22,1287.904275,19.315725


In [7]:
# Jan 2025 North Central: which single drop / add / swap reproduces the published 1,307.22?
jan = P[(P.vintage == dt.date(2025, 1, 1)) & (P.month == dt.date(2025, 1, 1))].set_index("state").price
nc = [s for s, z in ZONE.items() if z == "North Central"]
others = [s for s in jan.index if s not in nc]
cands = ([(f"drop {a}", [s for s in nc if s != a]) for a in nc] + [(f"add {b}", nc + [b]) for b in others]
         + [(f"swap {a}->{b}", [s for s in nc if s != a] + [b]) for a in nc for b in others])
[(name, round(jan[c].mean(), 4)) for name, c in cands if abs(jan[c].mean() - 1307.22) < 0.006]

[('drop Kwara', np.float64(1307.2222))]

## 6. Revisions across vintages (D7)

In [8]:
wide = P.pivot_table(index=["state", "month"], columns="vintage", values="price")
rows = []
for (s, m), row in wide.iterrows():
    v = row.dropna()
    for vin, val in list(v.items())[1:]:
        rows.append({"lag": (vin.year - m.year) * 12 + vin.month - m.month, "changed": abs(val - v.iloc[0]) > 0.005})
R = pd.DataFrame(rows)
R.groupby("lag").agg(overlaps=("changed", "size"), changed=("changed", "sum"))

,overlaps,changed
lag,,
1,666,0
12,296,0


## 7. Rounding (D4): 2 dp storage vs published figures, and Excel's 15-digit rounding

In [9]:
def half_up(x, digits=None):
    s = f"{x:.15g}" if digits == 15 else repr(float(x))
    return Decimal(s).quantize(Decimal("0.01"), ROUND_HALF_UP)

miss = []
for (vin, m), g in P.groupby(["vintage", "month"]):
    pub = half_up(N[(N.vintage == vin) & (N.month == m)].published.iloc[0])
    from_2dp = half_up(sum(half_up(x) for x in g.price) / 37)
    if from_2dp != pub:
        miss.append({"vintage": vin, "month": m, "published": pub, "mean_of_2dp_states": from_2dp})
print(f"{len(miss)} of {P.groupby(['vintage', 'month']).ngroups} published national figures missed by 2-dp storage")
v = 980.3549999999999  # Rivers, Sep 2025
print("Rivers Sep-25:", v, "| naive half-up:", half_up(v), "| Excel (15 sig. digits, then half-up):", half_up(v, 15))
pd.DataFrame(miss)

2 of 57 published national figures missed by 2-dp storage
Rivers Sep-25: 980.3549999999999 | naive half-up: 980.35 | Excel (15 sig. digits, then half-up): 980.36


,vintage,month,published,mean_of_2dp_states
0,2025-04-01,2024-04-01,701.24,701.25
1,2025-08-01,2024-08-01,830.46,830.45


## 8. Data quality by month

In [10]:
u = P.drop_duplicates(["month", "state"])
q = u.groupby("month").price.agg(whole_naira=lambda s: int((s % 1 == 0).sum()),
                                 max_min_ratio=lambda s: round(s.max() / s.min(), 2), min="min", max="max")
first = P[P.month == P.vintage].sort_values("month")
first = first.assign(mom=first.groupby("state").price.pct_change() * 100)
print("|MoM| > 40%:", first[first.mom.abs() > 40][["state", "month", "price", "mom"]].round(1).to_dict("records"))
print("price range across all observations:", round(P.price.min(), 2), "->", round(P.price.max(), 2))
q.round(2)

|MoM| > 40%: [{'state': 'Kaduna', 'month': datetime.date(2025, 1, 1), 'price': 1638.3, 'mom': 41.1}]
price range across all observations: 602.55 -> 1730.0


,whole_naira,max_min_ratio,min,max
month,,,,
2023-11-01,10,1.12,618.00,691.00
2023-12-01,7,1.29,602.78,776.54
2024-01-01,9,1.30,614.90,796.67
2024-02-01,11,1.15,650.00,750.43
2024-03-01,5,1.21,630.75,761.92
2024-04-01,9,1.32,602.55,797.78
2024-05-01,10,1.47,636.80,937.50
2024-06-01,5,1.36,626.94,854.55
2024-07-01,7,1.46,650.00,950.00


## 9. PDFs: release dates and the appendix table (D5)

In [11]:
def pdf_info(path):
    with pdfplumber.open(path) as pdf:
        cover = (pdf.pages[0].extract_text() or "").upper()
        appendix = [p for p in pdf.pages if "APPENDIX" in (p.extract_text() or "").upper()][-1]  # p.2 is the contents page
        table = max(appendix.extract_tables(), key=len)
        created = pdf.metadata.get("CreationDate", "")[2:10]
    states = [r for r in table[1:] if r[0] and not r[0].isupper() and r[0].strip() != "Grand Total"]
    return {"pdf_created": f"{created[:4]}-{created[4:6]}-{created[6:]}",
            "report_date": (re.search(r"REPORT DATE:\s*([A-Z]+ \d{4})", cover) or [None, None])[1],
            "appendix_page": appendix.page_number, "table_shape": f"{len(table)}x{len(table[0])}",
            "months": "/".join(c for c in table[0] if c and "-" in c), "state_rows": len(states),
            "uses_Nasarawa": any(r[0].strip() == "Nasarawa" for r in states)}

pdfs = pd.DataFrame([{"release": d.name, **pdf_info(next(d.glob("*.pdf")))} for d in RELEASES])
pdfs

,release,pdf_created,report_date,appendix_page,table_shape,months,state_rows,uses_Nasarawa
0,2024-11,2025-01-16,DECEMBER 2024,8,45x6,Oct-23/Sep-24/Oct-24,37,True
1,2024-12,2025-01-16,JANUARY 2025,8,45x6,Dec-23/Nov-24/Dec-24,37,True
2,2025-01,2025-03-25,FEBRUARY 2025,8,45x6,Jan-24/Dec-24/Jan-25,37,True
3,2025-02,2025-03-25,MARCH 2025,8,45x6,Feb-24/Jan-25/Feb-25,37,True
4,2025-03,2025-04-23,APRIL 2025,8,45x6,Mar-24/Feb-25/Mar-25,37,True
5,2025-04,2025-05-21,MAY 2025,8,45x6,Apr-24/Mar-25/Apr-25,37,True
6,2025-05,2025-06-30,JUNE 2025,8,45x6,May-24/Apr-25/May-25,37,True
7,2025-06,2025-07-31,JULY 2025,8,45x6,Jun-24/May-25/Jun-25,37,True
8,2025-07,2025-08-20,AUGUST 2025,8,45x6,Jul-24/Jun-25/Jul-25,37,True
9,2025-08,2026-02-25,SEPTEMBER 2025,8,42x6,Aug-24/Jul-25/Aug-25,35,True
